In [ ]:


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

import csv
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]

GOLD_06_DIR = (
    PROJECT_ROOT
    / "Gold"
    / "perguntas_negocio"
    / "gold_06_diferencas_regiao_senioridade_modelo"
)

OUTPUT_DIR = SCRIPT_DIR.parent / "outputs" / "gold_06"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# CARREGAR GOLD 06
"""
A análise mantém somente a variável de faixa salarial porque os cruzamentos posteriores têm como objetivo comparar a distribuição de remuneração entre senioridade, modelo de trabalho e região, sem misturar outras dimensões da Gold 06.
"""
# ---------------------------------------------------------------------

arquivos_gold_06 = [
    str(arquivo) for arquivo in GOLD_06_DIR.glob("part-*.csv")
]

print("\nGOLD 06:")
print(GOLD_06_DIR)

print("\nARQUIVOS ENCONTRADOS:")
print(arquivos_gold_06)

if not arquivos_gold_06:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {GOLD_06_DIR}"
    )

df_gold_06 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_06)
)

df = (
    df_gold_06
    .filter(F.col("variavel") == "faixa_salarial")
)


# ---------------------------------------------------------------------
# ORDENAR FAIXAS SALARIAIS
"""
As faixas salariais recebem uma ordem numérica crescente para permitir o cálculo da distribuição acumulada, da mediana e do P75. A remuneração continua sendo apresentada nas faixas originais da pesquisa, sem estimar salários pontuais.
"""
# ---------------------------------------------------------------------

df = (
    df.withColumn(
        "ordem_faixa",
        F.when(F.col("valor") == "Menos de R$ 1.000/mês", 1)
        .when(F.col("valor") == "de R$ 1.001/mês a R$ 2.000/mês", 2)
        .when(F.col("valor") == "de R$ 2.001/mês a R$ 3.000/mês", 3)
        .when(F.col("valor") == "de R$ 3.001/mês a R$ 4.000/mês", 4)
        .when(F.col("valor") == "de R$ 4.001/mês a R$ 6.000/mês", 5)
        .when(F.col("valor") == "de R$ 6.001/mês a R$ 8.000/mês", 6)
        .when(F.col("valor") == "de R$ 8.001/mês a R$ 12.000/mês", 7)
        .when(F.col("valor") == "de R$ 12.001/mês a R$ 16.000/mês", 8)
        .when(F.col("valor") == "de R$ 16.001/mês a R$ 20.000/mês", 9)
        .when(F.col("valor") == "de R$ 20.001/mês a R$ 25.000/mês", 10)
        .when(F.col("valor") == "de R$ 25.001/mês a R$ 30.000/mês", 11)
        .when(F.col("valor") == "de R$ 30.001/mês a R$ 40.000/mês", 12)
        .when(F.col("valor") == "Acima de R$ 40.001/mês", 13)
    )
)


# ---------------------------------------------------------------------
# ORDENAR SENIORIDADE
"""
A senioridade também recebe uma ordem explícita para manter Júnior, Pleno, Sênior e Especialista/Staff+ na sequência analítica esperada nos resultados dos cruzamentos.
"""
# ---------------------------------------------------------------------

df = (
    df.withColumn(
        "ordem_nivel",
        F.when(F.col("nivel") == "Júnior", 1)
        .when(F.col("nivel") == "Pleno", 2)
        .when(F.col("nivel") == "Sênior", 3)
        .when(F.col("nivel") == "Especialista/Staff+", 4)
        .otherwise(99)
    )
)


# ---------------------------------------------------------------------
# SIMPLIFICAR MODELO DE TRABALHO
"""
Os nomes dos modelos de trabalho são apenas simplificados para apresentação. As quatro categorias originais são preservadas conceitualmente, sem agrupamento entre modalidades distintas.
"""
# ---------------------------------------------------------------------

df = (
    df.withColumn(
        "modelo_trabalho",
        F.when(
            F.col("modelo_de_trabalho_atual") == "Modelo 100% remoto",
            "100% remoto"
        )
        .when(
            F.col("modelo_de_trabalho_atual") == "Modelo 100% presencial",
            "100% presencial"
        )
        .when(
            F.col("modelo_de_trabalho_atual")
            == "Modelo híbrido com dias fixos de trabalho presencial",
            "Híbrido com dias fixos"
        )
        .when(
            F.col("modelo_de_trabalho_atual")
            == "Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)",
            "Híbrido flexível"
        )
        .otherwise(F.col("modelo_de_trabalho_atual"))
    )
)


# ---------------------------------------------------------------------
# VALIDAR FAIXAS SALARIAIS
# ---------------------------------------------------------------------

faixas_sem_ordem = (
    df
    .filter(F.col("ordem_faixa").isNull())
    .select("valor")
    .distinct()
)

if faixas_sem_ordem.count() > 0:
    print("\nFAIXAS SALARIAIS NÃO MAPEADAS:")

    faixas_sem_ordem.show(
        100,
        truncate=False
    )

    raise ValueError(
        "Existem faixas salariais sem ordem definida."
    )

else:
    print(
        "\nTodas as faixas salariais foram classificadas corretamente."
    )
"""
A execução do notebook confirmou que nenhuma faixa salarial ficou sem ordem definida. Essa validação é importante porque qualquer faixa não mapeada comprometeria os percentis calculados posteriormente.
"""


# ---------------------------------------------------------------------
# MAPA DAS FAIXAS SALARIAIS
# ---------------------------------------------------------------------

FAIXAS_SALARIAIS = {
    1: "Menos de R$ 1.000/mês",
    2: "de R$ 1.001/mês a R$ 2.000/mês",
    3: "de R$ 2.001/mês a R$ 3.000/mês",
    4: "de R$ 3.001/mês a R$ 4.000/mês",
    5: "de R$ 4.001/mês a R$ 6.000/mês",
    6: "de R$ 6.001/mês a R$ 8.000/mês",
    7: "de R$ 8.001/mês a R$ 12.000/mês",
    8: "de R$ 12.001/mês a R$ 16.000/mês",
    9: "de R$ 16.001/mês a R$ 20.000/mês",
    10: "de R$ 20.001/mês a R$ 25.000/mês",
    11: "de R$ 25.001/mês a R$ 30.000/mês",
    12: "de R$ 30.001/mês a R$ 40.000/mês",
    13: "Acima de R$ 40.001/mês"
}

mapa_faixas = F.create_map(
    *[
        item
        for ordem, faixa in FAIXAS_SALARIAIS.items()
        for item in (
            F.lit(ordem),
            F.lit(faixa)
        )
    ]
)


# ---------------------------------------------------------------------
# FUNÇÃO PARA EXPORTAR CSV
# ---------------------------------------------------------------------

def exportar_csv(
    df_exportar,
    nome_arquivo
):
    caminho = OUTPUT_DIR / f"{nome_arquivo}.csv"

    linhas = df_exportar.collect()

    with open(
        caminho,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as arquivo:
        writer = csv.writer(
            arquivo,
            delimiter=";"
        )

        writer.writerow(df_exportar.columns)

        for linha in linhas:
            writer.writerow(
                [
                    linha[coluna]
                    for coluna in df_exportar.columns
                ]
            )

    print(f"\nArquivo salvo: {caminho}")


# ---------------------------------------------------------------------
# FUNÇÃO PARA CALCULAR P50 E P75 NOS CRUZAMENTOS
"""
A mesma função é reutilizada nos dois cruzamentos para garantir um critério único. O P50 é a primeira faixa que alcança 50% da distribuição acumulada e o P75 a primeira que alcança 75%, sempre calculados dentro de cada edição, nível e dimensão analisada.
"""
"""
O ranking é feito dentro de cada senioridade, usando primeiro o P50 e depois o P75. Combinações com menos de 30 respondentes são mantidas no resultado, mas recebem sinalização de amostra pequena para evitar interpretações equivalentes às dos grupos mais robustos.
"""
# ---------------------------------------------------------------------

def calcular_resumo_cruzamento(
    base,
    dimensao
):
    distribuicao = (
        base
        .groupBy(
            "edicao",
            "nivel",
            "ordem_nivel",
            dimensao,
            "ordem_faixa",
            "valor"
        )
        .agg(
            F.sum("contagem").alias("contagem")
        )
    )

    janela_total = (
        Window
        .partitionBy(
            "edicao",
            "nivel",
            dimensao
        )
    )

    janela_acumulada = (
        Window
        .partitionBy(
            "edicao",
            "nivel",
            dimensao
        )
        .orderBy("ordem_faixa")
        .rowsBetween(
            Window.unboundedPreceding,
            Window.currentRow
        )
    )

    distribuicao = (
        distribuicao
        .withColumn(
            "total_respondentes",
            F.sum("contagem").over(janela_total)
        )
        .withColumn(
            "acumulado",
            F.sum("contagem").over(janela_acumulada)
        )
        .withColumn(
            "pct_na_dimensao",
            F.round(
                (
                    F.col("contagem")
                    / F.col("total_respondentes")
                ) * 100,
                1
            )
        )
    )

    resumo = (
        distribuicao
        .groupBy(
            "edicao",
            "nivel",
            "ordem_nivel",
            dimensao
        )
        .agg(
            F.max("total_respondentes").alias("total_respondentes"),
            F.min(
                F.when(
                    F.col("acumulado")
                    >= F.col("total_respondentes") * 0.50,
                    F.col("ordem_faixa")
                )
            ).alias("ordem_faixa_mediana"),
            F.min(
                F.when(
                    F.col("acumulado")
                    >= F.col("total_respondentes") * 0.75,
                    F.col("ordem_faixa")
                )
            ).alias("ordem_faixa_p75")
        )
        .withColumn(
            "faixa_salarial_mediana",
            F.element_at(
                mapa_faixas,
                F.col("ordem_faixa_mediana")
            )
        )
        .withColumn(
            "faixa_salarial_p75",
            F.element_at(
                mapa_faixas,
                F.col("ordem_faixa_p75")
            )
        )
        .withColumn(
            "status_amostra",
            F.when(
                F.col("total_respondentes") < 30,
                "Amostra pequena (<30)"
            )
            .otherwise("OK")
        )
    )

    janela_ranking = (
        Window
        .partitionBy(
            "edicao",
            "nivel"
        )
        .orderBy(
            F.desc("ordem_faixa_mediana"),
            F.desc("ordem_faixa_p75")
        )
    )

    resumo = (
        resumo
        .withColumn(
            "ranking_no_nivel",
            F.dense_rank().over(janela_ranking)
        )
        .select(
            "edicao",
            "nivel",
            "ordem_nivel",
            dimensao,
            "total_respondentes",
            "faixa_salarial_mediana",
            "faixa_salarial_p75",
            "ordem_faixa_mediana",
            "ordem_faixa_p75",
            "ranking_no_nivel",
            "status_amostra"
        )
        .orderBy(
            "edicao",
            "ordem_nivel",
            "ranking_no_nivel",
            dimensao
        )
    )

    return resumo


# ---------------------------------------------------------------------
# SENIORIDADE X MODELO DE TRABALHO
# ---------------------------------------------------------------------

print("\n" + "=" * 140)
print("1. SENIORIDADE X MODELO DE TRABALHO")
print("=" * 140)

senioridade_modelo = calcular_resumo_cruzamento(
    df,
    "modelo_trabalho"
)

senioridade_modelo.show(
    100,
    truncate=False
)
"""
Os outputs mostram que o modelo presencial tende a ocupar posições salariais inferiores dentro da mesma senioridade. Em 2025-2026, por exemplo, Pleno remoto apresenta P50 de R$ 8.001 a R$ 12.000, enquanto o presencial fica em R$ 4.001 a R$ 6.000. Entre Sênior, remoto e híbrido flexível têm P50 de R$ 12.001 a R$ 16.000, acima do presencial e do híbrido com dias fixos, ambos em R$ 8.001 a R$ 12.000.
"""
"""
Para Especialista/Staff+ em 2025-2026, o remoto também aparece no topo, com P50 de R$ 16.001 a R$ 20.000 e P75 de R$ 25.001 a R$ 30.000, enquanto o presencial apresenta P50 de R$ 8.001 a R$ 12.000 e P75 de R$ 12.001 a R$ 16.000.
"""

exportar_csv(
    senioridade_modelo,
    "06_04_senioridade_x_modelo_trabalho"
)


# ---------------------------------------------------------------------
# SENIORIDADE X REGIÃO
# ---------------------------------------------------------------------

print("\n" + "=" * 140)
print("2. SENIORIDADE X REGIÃO")
print("=" * 140)

senioridade_regiao = calcular_resumo_cruzamento(
    df,
    "regiao_onde_mora"
)

senioridade_regiao.show(
    100,
    truncate=False
)
"""
O cruzamento por região mostra diferenças menores e menos uniformes do que as observadas por modelo de trabalho. Em 2025-2026, Júnior no Centro-oeste, Sudeste e Sul apresenta P50 de R$ 3.001 a R$ 4.000, enquanto Nordeste e Norte ficam em R$ 2.001 a R$ 3.000. Para Pleno, quatro regiões permanecem em R$ 6.001 a R$ 8.000 e apenas o Norte aparece abaixo, com R$ 4.001 a R$ 6.000.
"""
"""
As leituras regionais exigem cautela principalmente no Norte e em parte dos grupos de Especialista/Staff+, pois o notebook sinaliza várias combinações com menos de 30 respondentes. Em 2025-2026, por exemplo, Especialista/Staff+ no Centro-oeste e no Norte aparece com P50 de R$ 20.001 a R$ 25.000, mas com apenas 17 e 3 respondentes, respectivamente.
"""

exportar_csv(
    senioridade_regiao,
    "06_05_senioridade_x_regiao"
)